<a href="https://colab.research.google.com/github/AndrijaM06/car-price-prediction/blob/main/04_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pretprocesiranje podataka: cars.csv
- razdvajamo ulazne karakteristike (X) od ciljne promenljive (y)
- definišemo numeričke, nominalne i ordinalne kolone
- pravimo transformere za svaku grupu kolona
- povezujemo sve u jedan `ColumnTransformer`


## Učitavanje skupa podataka sa karakteristikama

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/AndrijaM06/car-price-prediction/main/data/cars_features.csv"
df = pd.read_csv(url)
df.shape

(55585, 17)

In [5]:
df.columns.tolist()

['make',
 'model',
 'price_usd',
 'year',
 'condition',
 'mileage_km',
 'fuel_type',
 'volume_cm3',
 'color',
 'transmission',
 'drive_unit',
 'segment',
 'car_age',
 'mileage_per_year',
 'engine_volume_liters',
 'is_newer_car',
 'is_high_mileage']

## Definisanje ciljne promenljive i ulaznih kolona

Ciljna promenljiva je `price_usd`. Ulazne kolone delimo u tri grupe:

- **numeričke** - već imaju brojčani oblik (originalne kolone + nove
  karakteristike iz inženjeringa karakteristika)
- **nominalne kategorijske** - kategorije bez prirodnog redosleda
  (npr. `make`, `fuel_type`, `color`)
- **ordinalne kategorijske** - kategorije sa prirodnim redosledom
  (`condition`: `for parts` < `with damage` < `with mileage`)

In [6]:
TARGET_COLUMN = "price_usd"

NUMERIC_FEATURES = [
    "year",
    "mileage_km",
    "volume_cm3",
    "car_age",
    "mileage_per_year",
    "engine_volume_liters",
    "is_newer_car",
    "is_high_mileage",
]

CATEGORICAL_FEATURES = [
    "make",
    "fuel_type",
    "color",
    "transmission",
    "drive_unit",
    "segment",
]

ORDINAL_FEATURES = [
    "condition",
]

CONDITION_ORDER = ["for parts", "with damage", "with mileage"]

all_feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES + ORDINAL_FEATURES
print("Broj ulaznih kolona:", len(all_feature_columns))

Broj ulaznih kolona: 15


## Razdvajanje ulaznih karakteristika i ciljne promenljive

In [7]:
X = df[all_feature_columns].copy()
y = df[TARGET_COLUMN].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (55585, 15)
y shape: (55585,)


In [8]:
X.isna().sum()[X.isna().sum() > 0]

,0
volume_cm3,47
engine_volume_liters,47
drive_unit,1861
segment,5192


**Zaključak:** Nedostajuće vrednosti i dalje postoje u `volume_cm3`,
`drive_unit` i `segment` - upravo njih ćemo rešiti kroz `SimpleImputer` u
narednim koracima.

## Transformer za numeričke kolone

Numeričke kolone prolaze kroz dva koraka:
1. **imputacija medijanom** - popunjava nedostajuće vrednosti (medijana je
   otpornija na ekstremne vrednosti od proseka)
2. **standardno skaliranje** - dovodi sve numeričke kolone na uporedivu
   skalu (bitno jer npr. `year` i `mileage_km` imaju veoma različite
   opsege vrednosti)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

## Transformer za nominalne kategorijske kolone

Kolone kao što su `make`, `fuel_type` ili `color` nemaju prirodan redosled,
pa koristimo `OneHotEncoder` koji svaku kategoriju pretvara u posebnu
binarnu kolonu.

In [10]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

categorical_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

## Transformer za ordinalnu kolonu `condition`

Kolona `condition` ima jasan redosled: automobil "za delove" je u lošijem
stanju od automobila "sa oštećenjem", koji je opet u lošijem stanju od
automobila koji je samo "korišćen" (`with mileage`). Zato koristimo
`OrdinalEncoder` sa eksplicitno definisanim redosledom, umesto
one-hot encodinga.

In [11]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[CONDITION_ORDER])),
])

ordinal_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OrdinalEncoder(categories=[['for parts', 'with damage',
                                             'with mileage']]))])

## Spajanje svih transformera pomoću `ColumnTransformer`

In [12]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ("ord", ordinal_transformer, ORDINAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['year', 'mileage_km', 'volume_cm3', 'car_age',
                                  'mileage_per_year', 'engine_volume_liters',
                                  'is_newer_car', 'is_high_mileage']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['make', 'fuel_type', 'color', 'transmission',
                                  'drive_unit', 'segment']),
                                ('ord',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['for '
                                                                              'parts',
                                                                              'with '
                                                                              'damage',
                                                                              'with '
                                                                              'mileage']]))]),
                                 ['condition'])])

## Provera da preprocessor radi

Testiramo `fit_transform` nad celim `X`, da vidimo koliko kolona
dobijamo nakon pretprocesiranja (one-hot encoding značajno povećava broj
kolona jer npr. `make` ima 96 kategorija).

In [13]:
X_transformed = preprocessor.fit_transform(X)

print("Oblik pre pretprocesiranja:", X.shape)
print("Oblik posle pretprocesiranja:", X_transformed.shape)

Oblik pre pretprocesiranja: (55585, 15)
Oblik posle pretprocesiranja: (55585, 136)


**Zaključak:** Od 15 originalnih ulaznih kolona dobili smo 136 kolona
nakon pretprocesiranja - većinu tog povećanja izaziva one-hot encoding
kolone `make` (96 jedinstvenih vrednosti).

## Zaključak

Napravili smo kompletan `ColumnTransformer` koji:
- popunjava nedostajuće vrednosti u numeričkim i kategorijskim kolonama
- skalira numeričke kolone
- kodira nominalne kategorijske kolone pomoću one-hot encodinga
- kodira ordinalnu kolonu `condition` uz očuvanje prirodnog redosleda

Ovaj `preprocessor` je sada spreman da se poveže sa regresionim modelom u
jedan `Pipeline`. To radimo u sledećem koraku: **treniranje prvog
regresionog modela**.